# Stage 3: Select Closers + Final Evaluation

Loads best params from **Stage 1** (openers) and **Stage 2** (leaders/followers),
then grid-searches over **closer** parameters.

**Measures:** all groups (openers + followers + closers) PnL on validation.

**Output:** final evaluation on train/val/test.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib import (
    load_trades,
    split_data,
    simulate_copy_pnl,
    print_results,
    print_wallet_stats,
    load_stage_result,
)
from polymarket_analysis.copy_groups import build_wallet_groups

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

## Load data

In [ ]:
df_full = load_trades()
df_train, df_val, df_test = split_data(df_full)

## Load previous stage results

In [ ]:
stage1 = load_stage_result(stage=1)
stage2 = load_stage_result(stage=2)

# Merge: stage2 overrides stage1 where overlap (e.g. both define opener params)
fixed_params = {**stage1, **stage2}

print("Fixed params from stages 1+2:")
for k, v in fixed_params.items():
    print(f"  {k}: {v}")

## Grid search: closer parameters

In [ ]:
param_grid = dict(
    min_closer_copyable_sell_pnl=[0.0, 50.0, 100.0, 200.0],
    min_closer_sell_count=[5, 10, 20],
)

keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Grid: {len(combos)} parameter combinations")

In [ ]:
best_pnl = -np.inf
best_params = None
best_groups = None
results_log = []

for i, vals in enumerate(combos):
    params = dict(zip(keys, vals))
    all_params = {
        **fixed_params,  # stage 1 + stage 2 best params
        **params,        # closer params being swept
    }

    t0 = time.time()
    try:
        g = build_wallet_groups(df_train, **all_params)
        vr = simulate_copy_pnl(
            df_val, g,
            time_window_minutes=all_params["time_window_minutes"],
            measure_groups=["openers", "followers", "closers"],
        )
        total = (
            vr["openers"]["total_copyable_pnl"]
            + vr["followers"]["total_copyable_pnl"]
            + vr["closers"]["total_copyable_pnl"]
        )
        elapsed = time.time() - t0

        results_log.append({
            **params,
            "total_pnl": total,
            "elapsed": elapsed,
            "openers_pnl": vr["openers"]["total_copyable_pnl"],
            "followers_pnl": vr["followers"]["total_copyable_pnl"],
            "closers_pnl": vr["closers"]["total_copyable_pnl"],
            "closers_wallets": vr["closers"]["wallet_count"],
            "closers_trades": vr["closers"]["trade_count"],
        })

        if total > best_pnl:
            best_pnl = total
            best_params = params
            best_groups = g

        if (i + 1) % 5 == 0 or i == len(combos) - 1:
            print(f"  [{i + 1}/{len(combos)}] best_pnl={best_pnl:.2f}  elapsed={elapsed:.1f}s")
    except Exception as e:
        print(f"  [{i + 1}/{len(combos)}] ERROR: {e}")

print(f"\n{'=' * 60}")
print(f"BEST PARAMS (val total pnl={best_pnl:.2f}):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

## Closer grid results

In [ ]:
opt_df = pd.DataFrame(results_log).sort_values("total_pnl", ascending=False)
print("All closer configs:")
opt_df

In [ ]:
if best_groups is not None:
    gdf = best_groups["closers"]
    print(f"Closers group: {len(gdf)} wallets")
    if not gdf.empty:
        cols = [c for c in ["wallet", "copyable_roi", "trade_roi", "copyable_pnl", "total_pnl", "trade_count"] if c in gdf.columns]
        print(gdf[cols].head(15).to_string())

## Final evaluation on all splits

In [ ]:
if best_groups is not None:
    all_best_params = {**fixed_params, **best_params}
    tw = all_best_params["time_window_minutes"]

    for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
        r = simulate_copy_pnl(df_split, best_groups, time_window_minutes=tw)
        print_results(f"{split_name} RESULTS (final)", r)
        print_wallet_stats(best_groups)
        print()

    print("Final best params:")
    for k, v in all_best_params.items():
        print(f"  {k}: {v}")

## Visualization

In [ ]:
if best_groups is not None:
    # PnL by group across splits
    split_data_map = {"TRAIN": df_train, "VAL": df_val, "TEST": df_test}
    group_names = ["openers", "followers", "closers"]

    rows = []
    for sn, ds in split_data_map.items():
        r = simulate_copy_pnl(ds, best_groups, time_window_minutes=tw)
        for gn in group_names:
            rows.append({"split": sn, "group": gn, "pnl": r[gn]["total_copyable_pnl"], "roi_pct": r[gn]["total_copyable_pnl"] / max(r[gn]["total_notional"], 1e-9) * 100})
    plot_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, label in [(axes[0], "pnl", "Copyable PnL ($)"), (axes[1], "roi_pct", "Copyable ROI (%)")]:
        pivot = plot_df.pivot(index="split", columns="group", values=metric).reindex(["TRAIN", "VAL", "TEST"])
        pivot.plot(kind="bar", ax=ax, rot=0)
        ax.set_title(label)
        ax.set_xlabel("")
        ax.axhline(0, color="black", linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
if best_groups is not None:
    for name in ["openers", "leaders", "followers", "closers"]:
        gdf = best_groups[name]
        print(f"\n{'=' * 50}")
        print(f"{name.upper()} ({len(gdf)} wallets)")
        print(f"{'=' * 50}")
        if not gdf.empty:
            print(gdf.head(10).to_string())